# 03 RAG Pipeline

Validate markdown-first annual report processing, header-level chunking, optional vector database ingestion, and retrieval inspection.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from IPython.display import Markdown, display
import pandas as pd

from backend import ingest_reports
from market_analyst.config.settings import load_settings
from market_analyst.repositories.vector_db import full_text_search, hybrid_search, vector_search
from market_analyst.services.rag import discover_reports
from market_analyst.telemetry import configure_notebook_logging

logger = configure_notebook_logging(run_name="03_rag_pipeline")
settings = load_settings()
reports = discover_reports(PROJECT_ROOT / "reports")
[(report.ticker, report.company_name, report.path.name) for report in reports]

## Run Configuration

Keep `PERSIST_TO_VECTOR_DB = False` while inspecting chunks. Set it to `True` when you want the notebook to embed and store chunks in Postgres/pgvector.

In [ ]:
MAX_PAGES = 5
CHUNK_SIZE = 1400
CHUNK_OVERLAP = 180
PERSIST_TO_VECTOR_DB = False
RESET_VECTOR_COLLECTION = False

result = ingest_reports(
    reports=reports,
    max_pages=MAX_PAGES,
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    persist=PERSIST_TO_VECTOR_DB,
    reset_collection=RESET_VECTOR_COLLECTION,
)

print(f"reports: {result.report_count}")
print(f"chunks: {result.chunk_count}")
print(f"vector ids: {len(result.vector_ids)}")
print(f"reports rows: {result.reports_rows}")

## Markdown Preview

In [ ]:
first_markdown = result.markdown_reports[0]
print(first_markdown.report.path.name, "pages:", first_markdown.page_count)
display(Markdown(first_markdown.markdown[:5000]))

## Header Chunk Output

In [ ]:
chunk_rows = [
    {
        "chunk_id": chunk.chunk_id,
        "ticker": chunk.metadata.get("ticker"),
        "page": chunk.metadata.get("page"),
        "section": chunk.metadata.get("section"),
        "heading_path": chunk.heading_path,
        "chars": len(chunk.content),
        "preview": chunk.content[:240].replace("\n", " "),
    }
    for chunk in result.chunks
]
chunks_df = pd.DataFrame(chunk_rows)
display(chunks_df.head(25))

In [ ]:
sample_chunk = result.chunks[min(1, len(result.chunks) - 1)]
display(Markdown(f"### {sample_chunk.heading_path}\n\n{sample_chunk.content[:3500]}"))

## Retrieval Smoke Test

In [ ]:
QUERY = "annual report"

if PERSIST_TO_VECTOR_DB:
    full_text_df = pd.DataFrame(full_text_search(settings, QUERY, limit=5))
    vector_df = pd.DataFrame(vector_search(settings, QUERY, limit=5))
    hybrid_df = pd.DataFrame(hybrid_search(settings, QUERY, limit=5))
    def show_results(label, df, columns):
        print(label)
        if df.empty:
            print("No results")
        else:
            display(df[[column for column in columns if column in df.columns]])

    show_results("Full-text", full_text_df, ["ticker", "company_name", "full_text_rank", "content"])
    show_results("Vector", vector_df, ["ticker", "company_name", "vector_distance", "content"])
    show_results("Hybrid RRF", hybrid_df, ["ticker", "company_name", "rrf_score", "full_text_rank", "vector_distance", "content"])
else:
    print("Set PERSIST_TO_VECTOR_DB = True and rerun ingestion to write embeddings and test retrieval.")

## Validation

In [ ]:
assert result.report_count > 0
assert result.chunk_count > 0
assert all(chunk.metadata.get("source_path") for chunk in result.chunks)
assert all(chunk.metadata.get("heading_path") for chunk in result.chunks)
assert all(chunk.content.strip() for chunk in result.chunks)
if PERSIST_TO_VECTOR_DB:
    assert len(result.vector_ids) == result.chunk_count
    assert result.reports_rows == result.chunk_count
print("RAG ingestion notebook validation passed")